In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [62]:
df = pd.read_csv("data/nordpool-lv.csv", parse_dates=True, index_col=0, usecols=[0, 2])
df = df[(df.index >= "2026-01-12 12:00") & (df.index < "2026-01-14")]
df = df.sort_index()
df.columns = ["nordpool"]

transport_price = 0.03962  # EUR/kWh
df["buy"] = df["nordpool"] + transport_price
df["sell"] = np.clip(df["nordpool"] - transport_price - 0.02, min=0.0)

df

,nordpool,buy,sell
ts_start,,,
2026-01-12 12:00:00,0.30878,0.34840,0.24916
2026-01-12 12:15:00,0.28548,0.32510,0.22586
2026-01-12 12:30:00,0.28565,0.32527,0.22603
2026-01-12 12:45:00,0.28641,0.32603,0.22679
2026-01-12 13:00:00,0.29561,0.33523,0.23599
...,...,...,...
2026-01-13 22:45:00,0.10905,0.14867,0.04943
2026-01-13 23:00:00,0.30450,0.34412,0.24488
2026-01-13 23:15:00,0.24079,0.28041,0.18117


In [ ]:
avg = pd.read_excel(
    "data/DailyPattern.xlsx", sheet_name="DayPatternHour", index_col=0, nrows=24
)
avg = avg["Consumption%"]
avg = avg.rename("avg")
avg = avg.sort_index()

df["avg"] = pd.merge(left=df["nordpool"], right=avg, left_on=df.index.hour, right_index=True)["avg"]
df

,nordpool,buy,sell,avg
ts_start,,,,
2026-01-12 12:00:00,0.30878,0.34840,0.24916,3.0
2026-01-12 12:15:00,0.28548,0.32510,0.22586,3.0
2026-01-12 12:30:00,0.28565,0.32527,0.22603,3.0
2026-01-12 12:45:00,0.28641,0.32603,0.22679,3.0
2026-01-12 13:00:00,0.29561,0.33523,0.23599,3.0
...,...,...,...,...
2026-01-13 22:45:00,0.10905,0.14867,0.04943,2.0
2026-01-13 23:00:00,0.30450,0.34412,0.24488,2.0
2026-01-13 23:15:00,0.24079,0.28041,0.18117,2.0


In [121]:
sol = pd.read_csv("data/solar_tmy.csv", skiprows=17, nrows=24 * 365)
sol.index = pd.to_datetime(sol["time(UTC)"], format="%Y%m%d:%H%M").rename("ts")
sol = pd.merge(
    left=pd.Series(sol["G(h)"], name="ghi"),
    right=pd.Series(df.index, name="ts"),
    left_on=sol.index.strftime("%m-%d %H:%M"),
    right_on=df.index.strftime("%m-%d %H:%M"),
)
sol = sol.set_index("ts")
sol = sol["ghi"].resample("15min").interpolate()

df["ghi"] = sol
df["ghi"] = df["ghi"].fillna(0)
df


,nordpool,buy,sell,avg,ghi
ts_start,,,,,
2026-01-12 12:00:00,0.30878,0.34840,0.24916,3.0,24.00
2026-01-12 12:15:00,0.28548,0.32510,0.22586,3.0,21.25
2026-01-12 12:30:00,0.28565,0.32527,0.22603,3.0,18.50
2026-01-12 12:45:00,0.28641,0.32603,0.22679,3.0,15.75
2026-01-12 13:00:00,0.29561,0.33523,0.23599,3.0,13.00
...,...,...,...,...,...
2026-01-13 22:45:00,0.10905,0.14867,0.04943,2.0,0.00
2026-01-13 23:00:00,0.30450,0.34412,0.24488,2.0,0.00
2026-01-13 23:15:00,0.24079,0.28041,0.18117,2.0,0.00


In [124]:
df.to_excel("input_data.xlsx")